# Metrics on the internal and external datasets

Each run's **Dice** and **NSD (1 mm)** on the MSD test split (internal) and on the external dataset, as a figure and a table.

- **Figure**: one panel per metric. Each dot is one case: **blue circles = MSD test** (39 cases), **orange diamonds = external** (50 hippocampi). The bar with a number is the mean.
- **Table**: n, mean, median and std for every run, dataset and metric.

The two datasets are not scored on the same labels:
- **MSD test**: labels are anterior/posterior, so each case's score is the mean of its anterior and posterior score (from `<run>/metrics_test.csv`, made by `hippo.evaluate`).
- **External**: labels are subfields, so the whole hippocampus is scored (from `<run>/external/metrics_external.csv`, made by `hippo.external`).

So compare runs **within** a dataset; the gap between the datasets mixes the new data with the different scoring.

Set `RUNS` in the next cell, then run all cells. A run without one of the CSVs is shown with the dataset it has. The last cell saves the figure and table to `runs/figures/`. Research use only, not for diagnosis.

In [ ]:
RUNS = ["runs/baseline", "runs/clean_labels", "runs/no_norm"]

# Work from the project root (so runs/ resolves) and import hippo from src/
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hippo").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from hippo import metrics_plot

%matplotlib inline
scores = {Path(r).name: s for r in RUNS if (s := metrics_plot.case_scores(Path(r)))}
missing = [r for r in RUNS if Path(r).name not in scores]
print(f"runs with metrics: {list(scores)}" + (f"; no metrics yet: {missing}" if missing else ""))

In [ ]:
fig = metrics_plot.plot(scores)
plt.show()

In [ ]:
# Mean ± std per run; one column per dataset and metric
rows = metrics_plot.summary_rows(scores)
cols = [(ds, m) for ds, *_ in metrics_plot.DATASETS for m, _ in metrics_plot.METRICS]
names = {"internal": "MSD test", "external": "External", "dice": "Dice", "nsd": "NSD"}
cell = {(r["run"], r["dataset"], r["metric"]): f"{r['mean']:.3f} ± {r['std']:.3f} (n={r['n']})" for r in rows}
lines = ["| run | " + " | ".join(f"{names[ds]} {names[m]}" for ds, m in cols) + " |", "|---" * (len(cols) + 1) + "|"]
lines += [f"| {run} | " + " | ".join(cell.get((run, ds, m), "–") for ds, m in cols) + " |" for run in scores]
display(Markdown("\n".join(lines)))

In [ ]:
# Save the figure (.png) and the full table (.csv: n, mean, median, std) to runs/figures/
metrics_plot.main(RUNS);